In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_16_try_0.pickle

trying: ['title_for_x_axis_cell20', 'title_for_x_axis', 'title_for_x_axis_cell22']
me:  8
me:  6
me:  10
trying: ['title_for_x_axis_cell20', 'title_for_x_axis', 'title_for_x_axis_cell22']
me:  8
me:  6
me:  10
trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  10
me:  10
trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  10
me:  10
trying: ['base_dir_2020']
me:  1
trying: ['responses_df_2017']
me:  10
trying: ['responses_df_2021']
me:  10
trying: ['file_name_2019']
me:  1
trying: ['file_name_2022']
me:  1
trying: ['combine_subset_of_data_from_multiple_years_20']
me:  8
trying: ['file_name_2018']
me:  1
trying: ['add_year_column_to_dataframes_20']
me:  8
trying: ['directory_cell8']
me:  1
trying: ['question_of_interest_cell20']
me:  8
trying: ['question_of_interest_cell23']
me:  11
trying: ['load_survey_data']
me:  1
trying: ['count_then_return_percent_20']
me:  8
trying: ['count_then_return_percent_23']
me:  11
trying: ['base_dir_2017']
me:  1
trying: ['percenta

me:  12
trying: ['USA_responses_df_2022']


me:  12
trying: ['question_of_interest_cell24']
me:  12
trying: ['percentages_cell24']
me:  12
trying: ['question_of_interest_cell21']
me:  9
trying: ['responses_in_order_cell21']
me:  9
trying: ['percentages_cell21']
me:  9
trying: ['count_then_return_percent_21']
me:  9
trying: ['grab_subset_of_data_25']
me:  13
trying: ['online_learning_platforms_df_2022']
me:  13
trying: ['count_then_return_percent_19']
me:  7
trying: ['question_of_interest_cell25']
me:  13
trying: ['percentages_cell19']
me:  7
trying: ['question_of_interest_cell19']
me:  7
trying: ['question_of_interest_cell27']
me:  15
trying: ['np']
me:  0
trying: ['online_learning_platforms_df_2022_cell27']
me:  15
trying: ['warnings']
me:  0
trying: ['grab_subset_of_data_27']
me:  15
trying: ['question_of_interest_cell18']
me:  6
trying: ['country_df_combined']
me:  6
trying: ['country_df_combined_cell18']
me:  6
trying: ['add_year_column_to_dataframes_18']
me:  6
trying: ['responses_df_2020']
me:  10
trying: ['question_name']

me:  0
trying: ['to_clean']
me:  16
trying: ['sns']
me:  0
trying: ['glob']
me:  0
trying: ['percentages_cell17']
me:  5
trying: ['responses_in_order_cell28']
me:  16
trying: ['responses_in_order']
me:  5
trying: ['pd']
me:  0
trying: ['count_then_return_percent_17']
me:  5
trying: ['question_of_interest']
me:  5
trying: ['responses_df_2019']
me:  1
trying: ['responses_df_2018']
me:  1
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['percentages']
me:  5
trying: ['create_dataframe_of_counts_16']
me:  4
trying: ['age_df_combined']
me:  8
trying: ['percentages_per_country_df']
me:  4
trying: ['add_year_column_to_dataframes_22']
me:  10
trying: ['factor']
me:  1
trying: ['directory']
me:  1
Declaring variable np
Declaring variable warnings
Declaring variable go
Declaring variable px
Declaring variable sns
Declaring variable glob
Declaring variable pd
Declaring variable base_dir_2020
Declaring variable file_name_2019
Declaring variable file_name_2022
Declaring variable file_name_20

In [3]:
%%RecordEvent
%%time
### cell 17 ###

# Optimized code

def count_percent(df, col, mapping=None):
    """
    Compute value percentages for a column, applying an optional replacement mapping.
    Returns a Series of percentages rounded to 1 decimal.
    """
    s = df[col]
    if mapping:
        s = s.replace(mapping)
    counts = s.value_counts(dropna=False)
    total = s.count()
    return (counts * 100 / total).round(1)


def combine_degrees(question, x_axis, include_2017=False):
    """
    For each year’s responses DataFrame, compute percentage distribution of `question`,
    normalize mis-encoded degree labels, and return a concatenated DataFrame with columns
    ['percentage', 'year', x_axis].
    """
    mapping = {
        'Masterâ\x80\x99s degree': "Master's degree",
        'Bachelorâ\x80\x99s degree': "Bachelor's degree"
    }
    # collect (year, DataFrame) in desired order
    year_sources = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017:
        year_sources.insert(0, ('2017', responses_df_2017))

    # build a single DataFrame with a 'year' column and the question responses
    df_list = []
    for year, df_src in year_sources:
        df_work = df_src if year != '2017' else df_src.rename(columns={'FormalEducation': question})
        df_list.append(pd.DataFrame({
            'year': year,
            question: df_work[question]
        }))
    df_all = pd.concat(df_list, ignore_index=True)

    # normalize mis-encoded labels
    df_all[question] = df_all[question].replace(mapping)

    # count per (year, response) including NaNs, and total non-null per year
    counts = df_all.groupby(['year', question], dropna=False).size()
    totals = df_all.groupby('year')[question].count()

    # compute percentages
    pct = (counts.div(totals, level='year') * 100).round(1)

    # format result and reorder columns
    pct = pct.rename_axis(['year', x_axis]).reset_index(name='percentage')
    return pct[['percentage', 'year', x_axis]]

# Parameters
question_of_interest_cell29 = (
    'What is the highest level of formal education that you have attained or plan to attain within the next 2 years?'
)
title_for_x_axis_cell29 = ''

# Combine 2017–2022 data
degree_df_combined = combine_degrees(
    question_of_interest_cell29,
    title_for_x_axis_cell29,
    include_2017=True
)

# Re-categorize into subset plus 'Other'
subset_of_degrees = ["Bachelor's degree", "Master's degree", 'Doctoral degree']
degree_df_combined_cell29 = (
    degree_df_combined
    .assign(**{
        title_for_x_axis_cell29: lambda df: df[title_for_x_axis_cell29]
            .where(df[title_for_x_axis_cell29].isin(subset_of_degrees), 'Other')
    })
    .sort_values(by=['year', 'percentage'], ascending=True)
)

degree_df_combined_cell29.info()


<class 'pandas.core.frame.DataFrame'>
Index: 47 entries, 3 to 42
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  47 non-null     float64
 1   year        47 non-null     object 
 2               47 non-null     object 
dtypes: float64(1), object(2)
memory usage: 1.5+ KB
CPU times: user 23.3 ms, sys: 0 ns, total: 23.3 ms
Wall time: 23.3 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_17_try_1.pickle

migration speed (bps): 747670630.7196082


---------------------------
variables to migrate:
question_of_interest_cell26 117
learning_platform_df_combined 5297884
combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26 144
responses_df_2019_cell10 16046533
pd 72
glob 72
base_dir_2020 155
question_name 90
file_name_2019 78
combine_subset_of_data_from_multiple_years_18 144
file_name_2022 81
count_then_return_percent_18 144
file_name_2018 76
question_name_alternate_cell18 70
directory_cell8 170
question_name_alternate 56
load_survey_data 144
subset_of_countries 184
base_dir_2017 155
title_for_x_axis 49
base_dir_2018 155
question_of_interest_cell18 90
base_dir_2022 155
country_df_combined 15454
file_name_2020 81
country_df_combined_cell18 15454
base_dir_2021 155
add_year_column_to_dataframes_18 144
file_name_2017 76
file_name_2021 81
responses_df_2019 16501304
responses_df_2018 32983443
age_df_combined_cell22 10611
responses_df_2022 25476025
replace_hyphen_with_en_dash 144
count_then_return_per

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2021', 'responses_df_2020', 'responses_df_2018', 'responses_df_2022', 'responses_df_2019']
Intermediate variables ['base_dir_2018', 'base_dir_2022', 'file_name_2018', 'directory', 'base_dir_2021', 'factor', 'file_name_2021', 'file_name_2022', 'file_name_2019', 'base_dir_2017', 'base_dir_2019', 'directory_cell8', 'file_name_2017', 'base_dir_2020', 'file_name_2020', 'responses_df_2017']
Future variables ['percentages', 'responses_df_2022_cell10', 'responses_df_2017']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2019', 'responses_df_2018', 'responses_df_2022']
Active variables ['responses_df_2019_cell10', 'responses_df_2022_cell10', 'responses_df_2022']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_df_2022_cell10', 'responses_d

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_17_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[17], f)


In [7]:
opt_output = Out.get(4)

In [ ]:
subset_of_degrees_opt = subset_of_degrees
responses_df_2021_opt = responses_df_2021
responses_df_2020_opt = responses_df_2020
title_for_x_axis_cell29_opt = title_for_x_axis_cell29
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_17.pickle
assert subset_of_degrees_opt == subset_of_degrees
assert title_for_x_axis_cell29_opt == title_for_x_axis_cell29

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
